# classifying heads

In [11]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from typing import Optional, Tuple, Union


ArrayLike = Union[np.ndarray, torch.Tensor]


def _to_numpy_2d(x: ArrayLike, embedding_dim: int = 768) -> np.ndarray:
    """Convert input to a 2D float32 numpy array with expected embedding dimension."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError("Expected a 2D array, got shape {}".format(x.shape))
    if x.shape[1] != embedding_dim:
        raise ValueError("Expected embedding size {}, got {}".format(embedding_dim, x.shape[1]))
    return x


def _quantile_threshold(scores: np.ndarray, contamination: float) -> float:
    """Set threshold so roughly `contamination` fraction becomes anomalous."""
    if not 0.0 < contamination < 1.0:
        raise ValueError("contamination must be in (0, 1)")
    return float(np.quantile(scores, 1.0 - contamination))


class KNNAnomalyHead:
    """Distance-based head: anomaly score = mean distance to k nearest normal samples."""

    def __init__(self, k: int = 10, contamination: float = 0.05):
        self.k = k
        self.contamination = contamination
        self.knn = None
        self.threshold_ = None
        self.n_neighbors_ = None

    def fit(self, x_normal: ArrayLike):
        x_normal = _to_numpy_2d(x_normal)
        if len(x_normal) < 2:
            raise ValueError("Need at least 2 normal samples for KNN")

        self.n_neighbors_ = min(self.k, len(x_normal))
        self.knn = NearestNeighbors(n_neighbors=self.n_neighbors_, metric="euclidean")
        self.knn.fit(x_normal)

        train_scores = self.score_samples(x_normal)
        self.threshold_ = _quantile_threshold(train_scores, self.contamination)
        return self

    def score_samples(self, x: ArrayLike) -> np.ndarray:
        if self.knn is None:
            raise RuntimeError("Call fit() before score_samples()")
        x = _to_numpy_2d(x)
        distances, _ = self.knn.kneighbors(x, n_neighbors=self.n_neighbors_)
        return distances.mean(axis=1).astype(np.float32)

    def predict(self, x: ArrayLike) -> np.ndarray:
        if self.threshold_ is None:
            raise RuntimeError("Call fit() before predict()")
        return (self.score_samples(x) > self.threshold_).astype(np.int64)


class MahalanobisAnomalyHead:
    """Density head: anomaly score = Mahalanobis distance to normal distribution."""

    def __init__(self, contamination: float = 0.05, use_shrinkage: bool = True, reg: float = 1e-6):
        self.contamination = contamination
        self.use_shrinkage = use_shrinkage
        self.reg = reg
        self.mean_ = None
        self.inv_cov_ = None
        self.threshold_ = None

    def fit(self, x_normal: ArrayLike):
        x_normal = _to_numpy_2d(x_normal)
        self.mean_ = x_normal.mean(axis=0)

        if self.use_shrinkage:
            cov = LedoitWolf().fit(x_normal).covariance_.astype(np.float32)
        else:
            cov = np.cov(x_normal, rowvar=False).astype(np.float32)

        cov = cov + np.eye(cov.shape[0], dtype=np.float32) * self.reg
        self.inv_cov_ = np.linalg.pinv(cov).astype(np.float32)

        train_scores = self.score_samples(x_normal)
        self.threshold_ = _quantile_threshold(train_scores, self.contamination)
        return self

    def score_samples(self, x: ArrayLike) -> np.ndarray:
        if self.mean_ is None or self.inv_cov_ is None:
            raise RuntimeError("Call fit() before score_samples()")
        x = _to_numpy_2d(x)

        delta = x - self.mean_
        md2 = np.einsum("bi,ij,bj->b", delta, self.inv_cov_, delta, optimize=True)
        md = np.sqrt(np.clip(md2, 0.0, None))
        return md.astype(np.float32)

    def predict(self, x: ArrayLike) -> np.ndarray:
        if self.threshold_ is None:
            raise RuntimeError("Call fit() before predict()")
        return (self.score_samples(x) > self.threshold_).astype(np.int64)


class _AE(nn.Module):
    def __init__(self, input_dim: int = 768, hidden_dim: int = 384, latent_dim: int = 128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return self.decoder(z)


class AutoEncoderAnomalyHead:
    """Reconstruction head: anomaly score = reconstruction MSE."""

    def __init__(
        self,
        contamination: float = 0.05,
        hidden_dim: int = 384,
        latent_dim: int = 128,
        weight_decay: float = 1e-6,
        device: Optional[str] = None,
    ):
        self.contamination = contamination
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.weight_decay = weight_decay
        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))

        self.model = None
        self.threshold_ = None

    def fit(
        self,
        x_normal: ArrayLike,
        epochs: int = 40,
        batch_size: int = 128,
        lr: float = 1e-3,
        verbose: bool = False,
    ):
        x_normal = _to_numpy_2d(x_normal)
        dataset = TensorDataset(torch.from_numpy(x_normal))
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

        self.model = _AE(input_dim=768, hidden_dim=self.hidden_dim, latent_dim=self.latent_dim).to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=self.weight_decay)

        self.model.train()
        for epoch in range(epochs):
            epoch_loss = 0.0
            for (xb,) in loader:
                xb = xb.to(self.device)
                recon = self.model(xb)
                loss = ((recon - xb) ** 2).mean()

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item() * len(xb)

            if verbose and (epoch + 1) % 10 == 0:
                print("[AE] epoch={:03d} loss={:.6f}".format(epoch + 1, epoch_loss / len(dataset)))

        train_scores = self.score_samples(x_normal)
        self.threshold_ = _quantile_threshold(train_scores, self.contamination)
        return self

    @torch.no_grad()
    def score_samples(self, x: ArrayLike) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Call fit() before score_samples()")
        x = _to_numpy_2d(x)

        self.model.eval()
        xt = torch.from_numpy(x).to(self.device)
        recon = self.model(xt)
        scores = ((recon - xt) ** 2).mean(dim=1)
        return scores.detach().cpu().numpy().astype(np.float32)

    def predict(self, x: ArrayLike) -> np.ndarray:
        if self.threshold_ is None:
            raise RuntimeError("Call fit() before predict()")
        return (self.score_samples(x) > self.threshold_).astype(np.int64)


class _MLPProjector(nn.Module):
    def __init__(self, input_dim: int = 768, hidden_dims: Tuple[int, ...] = (256, 128), rep_dim: int = 64):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, rep_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MLPOneClassAnomalyHead:
    """MLP one-class head (Deep SVDD style): score = distance to learned center."""

    def __init__(
        self,
        contamination: float = 0.05,
        hidden_dims: Tuple[int, ...] = (256, 128),
        rep_dim: int = 64,
        weight_decay: float = 1e-6,
        device: Optional[str] = None,
    ):
        self.contamination = contamination
        self.hidden_dims = hidden_dims
        self.rep_dim = rep_dim
        self.weight_decay = weight_decay
        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))

        self.model = None
        self.center_ = None
        self.threshold_ = None

    def _init_center(self, loader: DataLoader) -> torch.Tensor:
        self.model.eval()
        reps = []
        with torch.no_grad():
            for (xb,) in loader:
                xb = xb.to(self.device)
                reps.append(self.model(xb))

        center = torch.cat(reps, dim=0).mean(dim=0)

        # Prevent near-zero dimensions from collapsing gradients.
        eps = 1e-2
        center[(center.abs() < eps) & (center < 0)] = -eps
        center[(center.abs() < eps) & (center >= 0)] = eps
        return center.detach()

    def fit(
        self,
        x_normal: ArrayLike,
        epochs: int = 40,
        batch_size: int = 128,
        lr: float = 1e-3,
        verbose: bool = False,
    ):
        x_normal = _to_numpy_2d(x_normal)
        dataset = TensorDataset(torch.from_numpy(x_normal))
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

        self.model = _MLPProjector(input_dim=768, hidden_dims=self.hidden_dims, rep_dim=self.rep_dim).to(self.device)
        self.center_ = self._init_center(loader)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=self.weight_decay)

        self.model.train()
        for epoch in range(epochs):
            epoch_loss = 0.0
            for (xb,) in loader:
                xb = xb.to(self.device)
                z = self.model(xb)
                dist2 = ((z - self.center_) ** 2).sum(dim=1)
                loss = dist2.mean()

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item() * len(xb)

            if verbose and (epoch + 1) % 10 == 0:
                print("[MLP-SVDD] epoch={:03d} loss={:.6f}".format(epoch + 1, epoch_loss / len(dataset)))

        train_scores = self.score_samples(x_normal)
        self.threshold_ = _quantile_threshold(train_scores, self.contamination)
        return self

    @torch.no_grad()
    def score_samples(self, x: ArrayLike) -> np.ndarray:
        if self.model is None or self.center_ is None:
            raise RuntimeError("Call fit() before score_samples()")
        x = _to_numpy_2d(x)

        self.model.eval()
        xt = torch.from_numpy(x).to(self.device)
        z = self.model(xt)
        scores = ((z - self.center_) ** 2).sum(dim=1)
        return scores.detach().cpu().numpy().astype(np.float32)

    def predict(self, x: ArrayLike) -> np.ndarray:
        if self.threshold_ is None:
            raise RuntimeError("Call fit() before predict()")
        return (self.score_samples(x) > self.threshold_).astype(np.int64)


# Example usage:
# x_train_normal = ...  # shape: (N_train, 768), only normal sound embeddings
# x_eval = ...          # shape: (N_eval, 768)
#
# heads = {
#     "knn": KNNAnomalyHead(k=10, contamination=0.05),
#     "mahalanobis": MahalanobisAnomalyHead(contamination=0.05),
#     "autoencoder": AutoEncoderAnomalyHead(contamination=0.05),
#     "mlp_svdd": MLPOneClassAnomalyHead(contamination=0.05),
# }
#
# for name, head in heads.items():
#     head.fit(x_train_normal)
#     scores = head.score_samples(x_eval)  # higher score = more anomalous
#     preds = head.predict(x_eval)         # 1=anomaly, 0=normal
#     print(name, "threshold:", head.threshold_, "anomalies:", preds.sum())

In [12]:
import os

FEATURE_PATH = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\section_00_source_train_normal_0000_noAttribute.npy"

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError("Embedding file not found: {}".format(FEATURE_PATH))

x_raw = np.load(FEATURE_PATH)
print("Loaded raw embeddings:", x_raw.shape, x_raw.dtype)


def prepare_embedding_matrix(x: np.ndarray, embedding_dim: int = 768) -> np.ndarray:
    """Normalize loaded array into shape (N, 768) for head smoke testing."""
    x = np.asarray(x, dtype=np.float32)

    if x.ndim == 1:
        x = x[None, :]
    elif x.ndim >= 2:
        x = x.reshape(-1, x.shape[-1])

    if x.shape[1] != embedding_dim and x.shape[0] == embedding_dim:
        x = x.T

    if x.shape[1] != embedding_dim:
        raise ValueError("Expected final shape (N, 768), got {}".format(x.shape))

    return x


X = prepare_embedding_matrix(x_raw, embedding_dim=768)

if X.shape[0] < 2:
    X = np.repeat(X, 2, axis=0)
    print("Only one sample found, duplicated for smoke testing.")

rng = np.random.default_rng(42)
perm = rng.permutation(len(X))

if len(X) <= 2:
    train_idx = np.arange(len(X))
    eval_idx = np.arange(len(X))
else:
    n_train = max(2, int(0.8 * len(X)))
    n_train = min(n_train, len(X) - 1)
    train_idx = perm[:n_train]
    eval_idx = perm[n_train:]

x_train_normal = X[train_idx]
x_eval = X[eval_idx]

print("Prepared embeddings:", X.shape)
print("Train normal:", x_train_normal.shape, "| Eval:", x_eval.shape)

Loaded raw embeddings: (768,) float32
Only one sample found, duplicated for smoke testing.
Prepared embeddings: (2, 768)
Train normal: (2, 768) | Eval: (2, 768)


In [13]:
# Smoke test 1/4: KNN distance head
knn_head = KNNAnomalyHead(k=10, contamination=0.05)
knn_head.fit(x_train_normal)

knn_scores = knn_head.score_samples(x_eval)
knn_preds = knn_head.predict(x_eval)

print("[KNN] PASS")
print("threshold:", float(knn_head.threshold_))
print("score mean/std:", float(knn_scores.mean()), float(knn_scores.std()))
print("pred anomalies:", int(knn_preds.sum()), "/", len(knn_preds))

[KNN] PASS
threshold: 0.0
score mean/std: 0.0 0.0
pred anomalies: 0 / 2


In [14]:
# Smoke test 2/4: Mahalanobis density head
maha_head = MahalanobisAnomalyHead(contamination=0.05, use_shrinkage=True)
maha_head.fit(x_train_normal)

maha_scores = maha_head.score_samples(x_eval)
maha_preds = maha_head.predict(x_eval)

print("[Mahalanobis] PASS")
print("threshold:", float(maha_head.threshold_))
print("score mean/std:", float(maha_scores.mean()), float(maha_scores.std()))
print("pred anomalies:", int(maha_preds.sum()), "/", len(maha_preds))

[Mahalanobis] PASS
threshold: 0.0
score mean/std: 0.0 0.0
pred anomalies: 0 / 2


In [15]:
# Smoke test 3/4: AutoEncoder reconstruction head
ae_head = AutoEncoderAnomalyHead(contamination=0.05, hidden_dim=384, latent_dim=128)
ae_head.fit(
    x_train_normal,
    epochs=3,
    batch_size=min(128, len(x_train_normal)),
    lr=1e-3,
    verbose=False,
)

ae_scores = ae_head.score_samples(x_eval)
ae_preds = ae_head.predict(x_eval)

print("[AutoEncoder] PASS")
print("threshold:", float(ae_head.threshold_))
print("score mean/std:", float(ae_scores.mean()), float(ae_scores.std()))
print("pred anomalies:", int(ae_preds.sum()), "/", len(ae_preds))

[AutoEncoder] PASS
threshold: 0.04966169223189354
score mean/std: 0.04966169223189354 0.0
pred anomalies: 0 / 2


In [16]:
# Smoke test 4/4: MLP one-class (Deep SVDD style) head
mlp_head = MLPOneClassAnomalyHead(
    contamination=0.05,
    hidden_dims=(256, 128),
    rep_dim=64,
)
mlp_head.fit(
    x_train_normal,
    epochs=3,
    batch_size=min(128, len(x_train_normal)),
    lr=1e-3,
    verbose=False,
)

mlp_scores = mlp_head.score_samples(x_eval)
mlp_preds = mlp_head.predict(x_eval)

print("[MLP-SVDD] PASS")
print("threshold:", float(mlp_head.threshold_))
print("score mean/std:", float(mlp_scores.mean()), float(mlp_scores.std()))
print("pred anomalies:", int(mlp_preds.sum()), "/", len(mlp_preds))

[MLP-SVDD] PASS
threshold: 0.014739743433892727
score mean/std: 0.014739743433892727 0.0
pred anomalies: 0 / 2
